# 03 — Model Training

This notebook **fits and persists** every trainable recommender model. No evaluation
happens here — that lives in `04_evaluation.ipynb`, which loads these saved artifacts
and scores them on the held-out test set. Separating training from evaluation keeps the
expensive fitting apart from the cheap, frequently re-run metric computation.

**Protocol:**
- Surprise models (SVD) are tuned by 5-fold cross-validation on the training set;
  the weighted hybrid's α is chosen by exhaustive validation-set search.
- Every model is fit on the full training set and serialised to `artifacts/models/`.
- All random seeds are pinned (`RANDOM_STATE = 42`) so results are reproducible.

**Models trained:** Content-Based · User-kNN · Item-kNN · SVD ·
Weighted Hybrid · Stacked Hybrid. (The two naive baselines — Global Mean and
Popularity — are trivial to recompute, so they are defined directly in the
evaluation notebook rather than persisted here.)


In [ ]:
import sys
sys.path.insert(0, "../src")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import KFold

from hybrid_recsys.pipeline.data import load_processed
from hybrid_recsys.pipeline.splits import load_splits
from hybrid_recsys.pipeline.features import load_item_features
from hybrid_recsys.models.content import ContentBasedRecommender
from hybrid_recsys.models.collaborative import SVDModel, ItemKNNModel, UserKNNModel, _to_surprise
from hybrid_recsys.models.hybrid import WeightedHybrid, StackedHybrid
from hybrid_recsys.config import ARTIFACTS_MODELS, RANDOM_STATE
from surprise import SVD as SurpriseSVD

ARTIFACTS_MODELS.mkdir(parents=True, exist_ok=True)


## 1. Load Data

In [ ]:
movies                     = load_processed("movies")
train, val, test           = load_splits()
item_features, movie_index = load_item_features()

print(f"Train:      {len(train):>10,} ratings | {train['userId'].nunique():>7,} users")
print(f"Validation: {len(val):>10,} ratings | {val['userId'].nunique():>7,} users")
print(f"Item features: {item_features.shape}")

# User rating histories from the training set (needed by content model & hybrids)
user_ratings_map: dict = (
    train
    .groupby("userId")
    .apply(lambda df: dict(zip(df["movieId"], df["rating"])))
    .to_dict()
)

# Training-set global mean — reused by the stacked hybrid's NaN fallback.
global_mean = float(train["rating"].mean())
print(f"Global mean rating: {global_mean:.4f}")


## 2. Content-Based Model

The content-based model predicts the rating for item *j* by finding its top-L
most content-similar items via cosine similarity on the feature matrix, then
computing a mean-centred weighted average over the user's ratings on those items:

$$\hat{r}^{CB}_{u,j} = \bar{r}_u + \frac{\sum_{i \in N^L_u(j)} \text{sim}(i,j)\,(r_{u,i} - \bar{r}_u)}{\sum_{i} |\text{sim}(i,j)| + \varepsilon}$$


In [ ]:
cb = ContentBasedRecommender(n_neighbors=50)
cb.fit(item_features, movie_index)
cb.save()
print("Saved: artifacts/models/content_model.joblib")


## 3. User-Based k-NN

User-based CF identifies the *k* most similar users to the target user
(Pearson baseline similarity) and aggregates their ratings on the target item.
We use Surprise's `KNNWithMeans` with `user_based=True`. To bound memory, the
model samples 20K users before fitting — non-sampled users fall back to the
baseline prediction at scoring time.


In [ ]:
user_knn = UserKNNModel(k=80, min_k=5)
user_knn.fit(train)
user_knn.save()
print("Saved: artifacts/models/user_knn_model.joblib")


## 4. Item-Based k-NN

Item-based CF identifies the *k* most similar items to the target item and
aggregates the user's ratings on those items. Caps to the 15K most-rated items
before fitting; because ratings concentrate on popular titles, this cap barely
reduces coverage (unlike the user-side cap).


In [ ]:
item_knn = ItemKNNModel(k=80, min_k=5)
item_knn.fit(train)
item_knn.save()
print("Saved: artifacts/models/item_knn_model.joblib")


## 5. SVD — Matrix Factorisation

Surprise's SVD decomposes the rating matrix into user and item latent factor
matrices with global, user, and item bias terms:

$$\hat{r}_{ui} = \mu + b_u + b_i + q_i^\top p_u$$

Hyperparameters (`n_factors`, `n_epochs`, `lr_all`, `reg_all`) are selected
by 5-fold cross-validation on the training set, minimising RMSE. `random_state`
is pinned inside `SVDModel.tune`, so the factor initialisation is reproducible.


In [ ]:
svd = SVDModel()
svd.tune(train, param_grid={
    "n_factors": [50, 100, 200],
    "n_epochs":  [20, 40],
    "lr_all":    [0.002, 0.005],
    "reg_all":   [0.02, 0.05],
})
svd.fit(train)

print(f"Best params: {svd.best_params}")

svd.save()
print("Saved: artifacts/models/svd_model.joblib")


## 6. Weighted Hybrid

The weighted hybrid linearly interpolates the SVD and content-based predictions:

$$\hat{r}^{H1}_{ui} = \alpha \cdot \hat{r}^{SVD}_{ui} + (1-\alpha) \cdot \hat{r}^{CB}_{ui}$$

The scalar weight $\alpha$ is tuned on the validation set by exhaustive search
over $\alpha \in [0, 1]$ with step 0.05, minimising RMSE. When the content
model returns NaN (unknown item), the prediction falls back to SVD.


In [ ]:
weighted = WeightedHybrid()
weighted.set_models(svd, cb)

best_alpha = weighted.tune_alpha(val, user_ratings_map)
print(f"Best alpha (SVD weight): {best_alpha:.2f}")

weighted.save()
print("Saved: artifacts/models/weighted_hybrid.joblib")


## 7. Stacked Hybrid — Ridge Meta-Learner

The stacking hybrid learns the optimal combination of base-model predictions
from data, rather than fixing it as a scalar weight.

**Protocol (leak-free):**
1. Split the training set into 5 folds.
2. For each fold, train all base models on 4 folds and generate out-of-fold (OOF)
   predictions on the held-out fold — no model ever predicts on data it was trained on.
3. Train a Ridge regression meta-model on the OOF predictions plus side features
   (item popularity, user rating count, item rating count).
4. At test time, base models are retrained on the full training set; the meta-model
   combines their predictions.

> **Note:** The OOF loop trains 4 models × 5 folds. On MovieLens 25M this is
> computationally intensive. Reduce `N_OOF_FOLDS` or set `OOF_SAMPLE_FRAC < 1.0`
> to use a stratified training sample if runtime is a concern.


In [ ]:
N_OOF_FOLDS    = 5
OOF_SAMPLE_FRAC = 1.0   # set to e.g. 0.2 for a faster run

train_oof = (
    train.sample(frac=OOF_SAMPLE_FRAC, random_state=RANDOM_STATE)
    if OOF_SAMPLE_FRAC < 1.0
    else train
).reset_index(drop=True)

print(f"OOF training rows: {len(train_oof):,}  (frac={OOF_SAMPLE_FRAC})")

kf        = KFold(n_splits=N_OOF_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_preds = np.full((len(train_oof), 4), np.nan)

for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(train_oof)):
    print(f"  Fold {fold_idx + 1}/{N_OOF_FOLDS} ...", end=" ", flush=True)
    fold_tr  = train_oof.iloc[tr_idx]
    fold_val = train_oof.iloc[val_idx]

    fold_ur_map = (
        fold_tr.groupby("userId")
        .apply(lambda df: dict(zip(df["movieId"], df["rating"])))
        .to_dict()
    )

    fold_cb  = ContentBasedRecommender(n_neighbors=50).fit(item_features, movie_index)
    fold_uknn = UserKNNModel(k=80, min_k=5).fit(fold_tr)
    fold_iknn = ItemKNNModel(k=80, min_k=5).fit(fold_tr)
    fold_svd_model = SurpriseSVD(**svd.best_params)
    fold_svd_model.fit(_to_surprise(fold_tr).build_full_trainset())

    for i, row in enumerate(fold_val.itertuples()):
        ur = fold_ur_map.get(row.userId, {})
        oof_preds[val_idx[i], 0] = fold_cb.predict(ur, row.movieId)
        oof_preds[val_idx[i], 1] = fold_uknn.predict(row.userId, row.movieId)
        oof_preds[val_idx[i], 2] = fold_iknn.predict(row.userId, row.movieId)
        oof_preds[val_idx[i], 3] = fold_svd_model.predict(str(row.userId), str(row.movieId)).est

    print("done")

print("OOF predictions complete.")


In [ ]:
# Drop rows where ANY base model returned NaN (e.g. CB cold-start on a fold) —
# Ridge cannot fit NaN features and would crash otherwise.
nan_rows = np.isnan(oof_preds).any(axis=1)
print(f"OOF NaN rows dropped: {nan_rows.sum():,} / {len(oof_preds):,}")
oof_preds = oof_preds[~nan_rows]
train_oof = train_oof.loc[~nan_rows].reset_index(drop=True)
assert len(train_oof) == oof_preds.shape[0], "OOF row count mismatch after NaN drop"

# Side features (always computed from the FULL training set so they are stable
# regardless of OOF_SAMPLE_FRAC).
train_item_pop = train.groupby("movieId").size().to_dict()
train_user_cnt = train.groupby("userId").size().to_dict()
train_item_cnt = train.groupby("movieId").size().to_dict()


def meta_features(df: pd.DataFrame, base_preds: np.ndarray) -> np.ndarray:
    pop  = np.array([train_item_pop.get(m, 0) for m in df["movieId"]], dtype=float)
    ucnt = np.array([train_user_cnt.get(u, 0) for u in df["userId"]],  dtype=float)
    icnt = np.array([train_item_cnt.get(m, 0) for m in df["movieId"]], dtype=float)
    return np.column_stack([base_preds, pop, ucnt, icnt])


X_meta = meta_features(train_oof, oof_preds)
y_meta = train_oof["rating"].values

stacked = StackedHybrid(alpha=1.0)
stacked.fit(X_meta, y_meta)

print("Meta-model coefficients:")
for name, coef in zip(StackedHybrid.FEATURE_NAMES, stacked.meta.coef_):
    print(f"  {name:<22} {coef:+.4f}")

# Attach side features + global mean so the saved model scores standalone
# (the serving bundle and the evaluation notebook never reload the training frame).
stacked.set_side_features(
    item_popularity=train_item_pop,
    user_count=train_user_cnt,
    item_count=train_item_cnt,
    global_mean=global_mean,
)
stacked.save()
print("\nSaved: artifacts/models/stacked_hybrid.joblib")


## Conclusion

All six trainable models are fitted and persisted to `artifacts/models/`:
`content_model.joblib`, `user_knn_model.joblib`, `item_knn_model.joblib`,
`svd_model.joblib`, `weighted_hybrid.joblib`, `stacked_hybrid.joblib`.

The two naive baselines (Global Mean, Popularity) are trivial to recompute and
are defined directly in the evaluation notebook. Proceed to
**`04_evaluation.ipynb`** to score every model on the held-out test set.
